# Baseline Evaluation — Qwen2.5-1.5B (Before Any Fine-Tuning)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoodm2/LLM-Lab/blob/main/notebooks/00_baseline_eval.ipynb)

Captures the model's coding ability **before** any training.  
Results saved to `evals/00_baseline/` for comparison after SFT and GRPO.

**Benchmark**: 10 fixed HumanEval problems run identically at every phase.  
**Metrics**: pass@1, qualitative output quality, inference speed.

## 1. Environment Setup

In [ ]:
# Install dependencies (Colab)
!pip install -q transformers accelerate datasets bitsandbytes

In [ ]:
import torch
import json
import time
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Model

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Instruct variant — already chat-capable baseline
EVAL_DIR = Path("../evals/00_baseline")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Model loaded. Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Fixed Benchmark Problems

10 problems from HumanEval — used identically at every eval phase.

In [ ]:
# Fixed benchmark: 10 HumanEval problems (task_id, prompt, canonical_solution, test)
BENCHMARK = [
    {
        "task_id": "HumanEval/0",
        "prompt": "def has_close_elements(numbers: list, threshold: float) -> bool:\n    \"\"\"Check if any two numbers in the list are closer than threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    \"\"\"",
        "canonical_solution": "    for i in range(len(numbers)):\n        for j in range(i+1, len(numbers)):\n            if abs(numbers[i] - numbers[j]) < threshold:\n                return True\n    return False",
        "test": "assert has_close_elements([1.0, 2.0, 3.0], 0.5) == False\nassert has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3) == True\nassert has_close_elements([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False"
    },
    {
        "task_id": "HumanEval/1",
        "prompt": "def separate_paren_groups(paren_string: str) -> list:\n    \"\"\"Split a string of nested parentheses into separate groups.\n    >>> separate_paren_groups('( ) (( )) (( )( ))')\n    ['()', '(())', '(()())']\n    \"\"\"",
        "canonical_solution": "    result = []\n    current = []\n    depth = 0\n    for c in paren_string.replace(' ', ''):\n        if c == '(':\n            depth += 1\n            current.append(c)\n        elif c == ')':\n            depth -= 1\n            current.append(c)\n            if depth == 0:\n                result.append(''.join(current))\n                current = []\n    return result",
        "test": "assert separate_paren_groups('( ) (( )) (( )( ))') == ['()', '(())', '(()())']"
    },
    {
        "task_id": "HumanEval/3",
        "prompt": "def below_zero(operations: list) -> bool:\n    \"\"\"Return True if balance drops below zero at any point.\n    >>> below_zero([1, 2, 3])\n    False\n    >>> below_zero([1, 2, -4, 5])\n    True\n    \"\"\"",
        "canonical_solution": "    balance = 0\n    for op in operations:\n        balance += op\n        if balance < 0:\n            return True\n    return False",
        "test": "assert below_zero([1, 2, 3]) == False\nassert below_zero([1, 2, -4, 5]) == True"
    },
    {
        "task_id": "HumanEval/4",
        "prompt": "def mean_absolute_deviation(numbers: list) -> float:\n    \"\"\"Calculate mean absolute deviation around the mean.\n    >>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])\n    1.0\n    \"\"\"",
        "canonical_solution": "    mean = sum(numbers) / len(numbers)\n    return sum(abs(x - mean) for x in numbers) / len(numbers)",
        "test": "assert abs(mean_absolute_deviation([1.0, 2.0, 3.0, 4.0]) - 1.0) < 1e-6"
    },
    {
        "task_id": "HumanEval/5",
        "prompt": "def intersperse(numbers: list, delimeter: int) -> list:\n    \"\"\"Insert delimeter between every two consecutive elements.\n    >>> intersperse([], 4)\n    []\n    >>> intersperse([1, 2, 3], 4)\n    [1, 4, 2, 4, 3]\n    \"\"\"",
        "canonical_solution": "    result = []\n    for i, n in enumerate(numbers):\n        result.append(n)\n        if i < len(numbers) - 1:\n            result.append(delimeter)\n    return result",
        "test": "assert intersperse([], 4) == []\nassert intersperse([1, 2, 3], 4) == [1, 4, 2, 4, 3]"
    },
    {
        "task_id": "HumanEval/6",
        "prompt": "def parse_nested_parens(paren_string: str) -> list:\n    \"\"\"Return max nesting depth for each group in paren_string.\n    >>> parse_nested_parens('(()()) ((())) () ((())()())')\n    [2, 3, 1, 3]\n    \"\"\"",
        "canonical_solution": "    def max_depth(s):\n        depth = max_d = 0\n        for c in s:\n            if c == '(': depth += 1; max_d = max(max_d, depth)\n            elif c == ')': depth -= 1\n        return max_d\n    return [max_depth(g) for g in paren_string.split()]",
        "test": "assert parse_nested_parens('(()()) ((())) () ((())()())') == [2, 3, 1, 3]"
    },
    {
        "task_id": "HumanEval/7",
        "prompt": "def filter_by_substring(strings: list, substring: str) -> list:\n    \"\"\"Filter strings containing the given substring.\n    >>> filter_by_substring([], 'a')\n    []\n    >>> filter_by_substring(['abc', 'bacd', 'cde', 'array'], 'a')\n    ['abc', 'bacd', 'array']\n    \"\"\"",
        "canonical_solution": "    return [s for s in strings if substring in s]",
        "test": "assert filter_by_substring(['abc', 'bacd', 'cde', 'array'], 'a') == ['abc', 'bacd', 'array']"
    },
    {
        "task_id": "HumanEval/9",
        "prompt": "def rolling_max(numbers: list) -> list:\n    \"\"\"Return running maximum element found until given moment in sequence.\n    >>> rolling_max([1, 2, 3, 2, 3, 4, 2])\n    [1, 2, 3, 3, 3, 4, 4]\n    \"\"\"",
        "canonical_solution": "    result = []\n    current_max = None\n    for n in numbers:\n        current_max = n if current_max is None else max(current_max, n)\n        result.append(current_max)\n    return result",
        "test": "assert rolling_max([1, 2, 3, 2, 3, 4, 2]) == [1, 2, 3, 3, 3, 4, 4]"
    },
    {
        "task_id": "HumanEval/11",
        "prompt": "def string_xor(a: str, b: str) -> str:\n    \"\"\"XOR two binary strings.\n    >>> string_xor('010', '110')\n    '100'\n    \"\"\"",
        "canonical_solution": "    return ''.join('0' if i == j else '1' for i, j in zip(a, b))",
        "test": "assert string_xor('010', '110') == '100'"
    },
    {
        "task_id": "HumanEval/12",
        "prompt": "def longest(strings: list) -> str:\n    \"\"\"Return longest string, or None if list is empty.\n    >>> longest([])\n    >>> longest(['a', 'b', 'c'])\n    'a'\n    >>> longest(['a', 'bb', 'ccc'])\n    'ccc'\n    \"\"\"",
        "canonical_solution": "    if not strings: return None\n    return max(strings, key=len)",
        "test": "assert longest([]) is None\nassert longest(['a', 'bb', 'ccc']) == 'ccc'"
    },
]

## 4. Inference Helper

In [ ]:
def generate_completion(prompt: str, max_new_tokens: int = 256) -> tuple[str, float]:
    """Run model on a coding prompt, return (completion, latency_seconds)."""
    messages = [
        {"role": "system", "content": "Complete the Python function. Return only the function body, no explanation."},
        {"role": "user", "content": prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy — deterministic for fair comparison
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - t0

    # Decode only the generated tokens (not the prompt)
    generated = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True), latency

## 5. Run Benchmark & Compute pass@1

In [ ]:
def execute_test(completion: str, problem: dict) -> bool:
    """Execute generated code + test cases. Returns True if all tests pass."""
    code = problem["prompt"] + "\n" + completion + "\n" + problem["test"]
    try:
        exec(code, {})
        return True
    except Exception:
        return False


results = []
for problem in BENCHMARK:
    completion, latency = generate_completion(problem["prompt"])
    passed = execute_test(completion, problem)
    results.append({
        "task_id": problem["task_id"],
        "passed": passed,
        "latency": round(latency, 2),
        "completion": completion,
        "canonical": problem["canonical_solution"],
    })
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {problem['task_id']}  ({latency:.2f}s)")

pass_at_1 = sum(r["passed"] for r in results) / len(results)
print(f"\npass@1: {pass_at_1:.0%}  ({sum(r['passed'] for r in results)}/{len(results)})")

## 6. Save Results

In [ ]:
summary = {
    "phase": "00_baseline",
    "model": MODEL_ID,
    "pass_at_1": pass_at_1,
    "num_passed": sum(r["passed"] for r in results),
    "num_total": len(results),
    "avg_latency": round(sum(r["latency"] for r in results) / len(results), 2),
    "results": results,
}

with open(EVAL_DIR / "results.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved to {EVAL_DIR / 'results.json'}")

## 7. Qualitative Inspection

Side-by-side: model output vs canonical solution for each problem.

In [ ]:
for r in results:
    status = "✓" if r["passed"] else "✗"
    print(f"{'='*60}")
    print(f"{status} {r['task_id']}")
    print(f"--- Model output ---")
    print(r["completion"])
    print(f"--- Canonical ---")
    print(r["canonical"])

# Save samples to markdown for easy reading
lines = ["# Baseline Samples\n"]
for r in results:
    status = "PASS" if r["passed"] else "FAIL"
    lines.append(f"## {r['task_id']} — {status}\n")
    lines.append(f"**Model:**\n```python\n{r['completion']}\n```\n")
    lines.append(f"**Canonical:**\n```python\n{r['canonical']}\n```\n")

with open(EVAL_DIR / "samples.md", "w") as f:
    f.write("\n".join(lines))
print(f"Samples saved to {EVAL_DIR / 'samples.md'}")

## 8. Summary Plot

In [ ]:
import matplotlib.pyplot as plt

labels = [r["task_id"].split("/")[1] for r in results]
colors = ["#2ecc71" if r["passed"] else "#e74c3c" for r in results]
latencies = [r["latency"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pass/fail per problem
axes[0].bar(labels, [1]*len(results), color=colors)
axes[0].set_title(f"pass@1 per problem — {pass_at_1:.0%} overall")
axes[0].set_ylabel("pass (green) / fail (red)")
axes[0].set_xticklabels(labels, rotation=45)

# Latency per problem
axes[1].bar(labels, latencies, color="#3498db")
axes[1].set_title("Inference latency per problem (s)")
axes[1].set_ylabel("seconds")
axes[1].set_xticklabels(labels, rotation=45)

plt.tight_layout()
plt.savefig(EVAL_DIR / "baseline_results.png", dpi=150)
plt.show()
print(f"Plot saved to {EVAL_DIR / 'baseline_results.png'}")